In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from google.colab import files
import os

# ==============================================================================
# 1. ЗАГРУЗКА ДАННЫХ
# ==============================================================================
print("Загрузите файлы train.csv и test.csv:")
uploaded = files.upload()

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Задаем имена целевой колонки и ID под новый турнир
target_col = 'satisfaction'
id_col = 'id'


Загрузите файлы train.csv и test.csv:


Saving test.csv to test (1).csv
Saving train.csv to train (1).csv


In [ ]:
# Объединяем датасеты для правильной генерации признаков
df = pd.concat([train, test], axis=0, ignore_index=True)
df


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999474,999474,32,1725,3,1,3,3,3,3,3,...,3,1,3,0,0.0,Female,Loyal Customer,Business travel,Business,NaN
999475,999475,24,1325,3,3,3,4,3,3,3,...,3,5,3,0,0.0,Female,disloyal Customer,Business travel,Eco,NaN
999476,999476,33,190,4,5,4,3,3,4,3,...,2,2,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,NaN
999477,999477,59,678,3,5,3,2,4,3,5,...,4,4,5,4,0.0,Male,Loyal Customer,Personal Travel,Eco,NaN


In [ ]:
# ==========================================
# МАГИЯ FEATURES ENGINEERING (РЕЙС КАК ID)
# ==========================================

# 2. Превращаем дистанцию в категориальный ID маршрута
# Переводим в строку, чтобы деревянные модели воспринимали это как категории, а не числа
df['route_id'] = df['Flight Distance'].astype(str)

# 3. Находим все колонки, связанные с оценками сервиса (обычно это шкала от 0 до 5)
# Например: Inflight wifi service, Ease of Online booking, Cleanliness и т.д.
service_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64']
                and col not in ['id', 'Flight Distance', 'Age', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'satisfaction']]

print(f"Найдено {len(service_cols)} колонок сервиса для агрегации по маршрутам.")

# 4. Считаем средний уровень комфорта для каждого конкретного маршрута
# Используем только данные из train, чтобы избежать утечки данных (Data Leakage)
train_df = df[df['satisfaction'].notna()]

for col in service_cols:
    # Считаем среднее значение сервиса для каждой дистанции
    route_service_mean = train_df.groupby('Flight Distance')[col].mean().to_dict()

Найдено 13 колонок сервиса для агрегации по маршрутам.


In [ ]:


# ==========================================
# МАГИЯ FEATURES ENGINEERING (РЕЙС КАК ID)
# ==========================================

# 2. Превращаем дистанцию в категориальный ID маршрута
# Переводим в строку, чтобы деревянные модели воспринимали это как категории, а не числа
df['route_id'] = df['Flight Distance'].astype(str)

# 3. Находим все колонки, связанные с оценками сервиса (обычно это шкала от 0 до 5)
# Например: Inflight wifi service, Ease of Online booking, Cleanliness и т.д.
service_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64']
                and col not in ['id', 'Flight Distance', 'Age', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'satisfaction']]

print(f"Найдено {len(service_cols)} колонок сервиса для агрегации по маршрутам.")

# 4. Считаем средний уровень комфорта для каждого конкретного маршрута
# Используем только данные из train, чтобы избежать утечки данных (Data Leakage)
train_df = df[df['satisfaction'].notna()]

for col in service_cols:
    # Считаем среднее значение сервиса для каждой дистанции
    route_service_mean = train_df.groupby('Flight Distance')[col].mean().to_dict()

    # Маппим эти средние значения как на train, так и на test
    # Если в test встретится новая дистанция, fillna(df[col].mean()) подставит глобальное среднее
    df[f'route_avg_{col}'] = df['Flight Distance'].map(route_service_mean).fillna(df[col].mean())

# 5. Разделяем датасеты обратно
X_train = df[df['satisfaction'].notna()].copy()
y_train = X_train['satisfaction']
X_train = X_train.drop(columns=['id', 'satisfaction'])

X_test = df[df['satisfaction'].isna()].copy()
X_test = X_test.drop(columns=['id', 'satisfaction'])

# 6. Указываем, какие колонки теперь категориальные
categorical_features = ['route_id']
# Добавьте сюда другие текстовые колонки из турнира, если они есть (например, 'Type of Travel', 'Class')
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')

print("\nПодготовка завершена!")
print(f"Количество признаков увеличилось с {train.shape[1]-2} до {X_train.shape[1]}")
print(f"Категориальные признаки: {list(X_train.select_dtypes(include=['category']).columns)}")


Найдено 13 колонок сервиса для агрегации по маршрутам.

Подготовка завершена!
Количество признаков увеличилось с 21 до 35
Категориальные признаки: ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'route_id']


In [ ]:
from scipy.stats import entropy

# Предполагаем, что df — это ваш объединенный датасет (train + test)

print("Применяем продвинутые математические методы...")

# ==========================================================
# 1. Энтропия Шеннона (Оценка эмоционального хаоса в строке)
# ==========================================================
# Выбираем только колонки с оценками сервиса (обычно от 0 до 5)
service_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64']
                and col not in ['id', 'Flight Distance', 'Age', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'satisfaction']]

def calculate_row_entropy(row):
    # Считаем, как часто встречается каждая оценка (от 0 до 5) в конкретной строке
    counts = np.bincount(row.astype(int), minlength=6)
    # Переводим в вероятности
    probs = counts / counts.sum()
    # Считаем чистую энтропию Шеннона
    return entropy(probs, base=2)

# Применяем по строкам (axis=1)
df['shannon_entropy'] = df[service_cols].apply(calculate_row_entropy, axis=1)


# ==========================================================
# 2. Нелинейное сглаживание задержек (Психологическое восприятие времени)
# ==========================================================
# Задержка прибытия часто содержит пропуски (NaN), заполним их нулями
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(0)

# Применяем затухающую функцию (аналог релятивистского фактора из физики)
# Чем больше задержка, тем ближе значение к 0. Падение от 0 до 10 минут будет резким, а от 100 до 200 — плавным.
df['delay_factor'] = 1.0 / np.sqrt(1.0 + (df['Arrival Delay in Minutes'] + df['Departure Delay in Minutes']) / 10.0)


# ==========================================================
# 3. Выделение ключевых парных взаимодействий (Feature Interactions)
# ==========================================================
# Перемножаем синергетические сервисы (например, интернет + развлечения в полете)
if 'Inflight wifi service' in df.columns and 'Inflight entertainment' in df.columns:
    df['digital_synergy'] = df['Inflight wifi service'] * df['Inflight entertainment']

if 'Seat comfort' in df.columns and 'Leg room service' in df.columns:
    df['physical_comfort_synergy'] = df['Seat comfort'] * df['Leg room service']

print(f"Математические признаки успешно добавлены! Текущий размер датасета: {df.shape}")


Применяем продвинутые математические методы...
Математические признаки успешно добавлены! Текущий размер датасета: (999479, 41)


In [ ]:
# Принудительно обновляем матрицы X_train и X_test из нового df
X_train = df[df['satisfaction'].notna()].copy()
X_test = df[df['satisfaction'].isna()].copy()

# Убираем колонки-ориентиры
X_train = X_train.drop(columns=['id', 'satisfaction'], errors='ignore')
X_test = X_test.drop(columns=['id', 'satisfaction'], errors='ignore')

# На всякий случай проверяем, что размерность совпадает (должно быть 39 признаков, так как 'id' и 'satisfaction' ушли)
print(f"Новая размерность X_train для бустингов: {X_train.shape}")


Новая размерность X_train для бустингов: (699635, 39)


In [ ]:
# Принудительно переводим все текстовые колонки в тип 'category' для XGBoost и CatBoost
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')

print(f"Категориальные колонки подготовлены: {list(X_train.select_dtypes(include=['category']).columns)}")


Категориальные колонки подготовлены: ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'route_id']


In [ ]:
# Синхронизируем категории между Train и Test, чтобы XGBoost не падал
categorical_cols = list(X_train.select_dtypes(include=['category']).columns)

for col in categorical_cols:
    # Собираем абсолютно все уникальные категории из обеих выборок
    union_categories = list(set(X_train[col].astype(str).unique()) | set(X_test[col].astype(str).unique()))

    # Перевыпускаем тип 'category' с полным списком возможных значений
    X_train[col] = pd.Categorical(X_train[col].astype(str), categories=union_categories)
    X_test[col] = pd.Categorical(X_test[col].astype(str), categories=union_categories)

print("Категории Train и Test идеально синхронизированы!")


Категории Train и Test идеально синхронизированы!


In [ ]:
import xgboost as xgb

# Параметры XGBoost для максимизации AUC
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',  # Быстрый гистограммный метод
    'enable_categorical': True,  # Важно для работы с route_id без One-Hot
    'random_state': 42,
    'n_jobs': -1
}

oof_preds_xgb = np.zeros(len(X_train))
test_preds_xgb = np.zeros(len(X_test))

print("Начало обучения XGBoost на 5 фолдах...")

for fold, (train_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]

    model_xgb = xgb.XGBClassifier(**xgb_params, n_estimators=3000, early_stopping_rounds=100)

    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=200
    )

    oof_preds_xgb[val_idx] = model_xgb.predict_proba(X_va)[:, 1]
    test_preds_xgb += model_xgb.predict_proba(X_test)[:, 1] / folds.n_splits

    fold_auc = roc_auc_score(y_va, oof_preds_xgb[val_idx])
    print(f"-> XGBoost Фолд {fold + 1} завершен. AUC: {fold_auc:.5f}")

cv_auc_xgb = roc_auc_score(y_train, oof_preds_xgb)
print(f"\nИтоговый OOF AUC для XGBoost: {cv_auc_xgb:.5f}")


Начало обучения XGBoost на 5 фолдах...
[0]	validation_0-auc:0.94175
[200]	validation_0-auc:0.95780
[400]	validation_0-auc:0.95865
[600]	validation_0-auc:0.95904
[800]	validation_0-auc:0.95923
[1000]	validation_0-auc:0.95930
[1063]	validation_0-auc:0.95931
-> XGBoost Фолд 1 завершен. AUC: 0.95932
[0]	validation_0-auc:0.93962
[200]	validation_0-auc:0.95700
[400]	validation_0-auc:0.95781
[600]	validation_0-auc:0.95814
[800]	validation_0-auc:0.95832
[1000]	validation_0-auc:0.95838
[1055]	validation_0-auc:0.95837
-> XGBoost Фолд 2 завершен. AUC: 0.95841
[0]	validation_0-auc:0.94170
[200]	validation_0-auc:0.95837
[400]	validation_0-auc:0.95931
[600]	validation_0-auc:0.95975
[800]	validation_0-auc:0.95997
[1000]	validation_0-auc:0.96013
[1067]	validation_0-auc:0.96013
-> XGBoost Фолд 3 завершен. AUC: 0.96014
[0]	validation_0-auc:0.94210
[200]	validation_0-auc:0.95797
[400]	validation_0-auc:0.95883
[600]	validation_0-auc:0.95909
[800]	validation_0-auc:0.95925
[1000]	validation_0-auc:0.95933
[1

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import io

# ==========================================
from sklearn.preprocessing import LabelEncoder

# ==========================================
# 1. АВТОМАТИЧЕСКОЕ ИСПРАВЛЕНИЕ ТАРГЕТА БЕЗ ХАРДКОДА
# ==========================================

# Выделяем train из общего df
X_train = df[df['satisfaction'].notna()].copy()

# Гарантированно кодируем текстовые классы в 0 и 1 через LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(X_train['satisfaction'].astype(str))

print("--- ПРОВЕРКА КОРРЕКТНОСТИ ТАРГЕТА ---")
print(f"Оригинальные классы, которые нашел энкодер: {le.classes_}")
print(f"Распределение закодированных классов в y_train: {np.bincount(y_train)}")
print("-------------------------------------\n")

# Убираем лишние колонки из признаков
X_train = X_train.drop(columns=['id', 'satisfaction'], errors='ignore')
X_test = df[df['satisfaction'].isna()].copy().drop(columns=['id', 'satisfaction'], errors='ignore')

# Приводим текстовые фичи к типу 'category'
categorical_features = ['route_id']
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')


# ==========================================
# 2. ОБУЧЕНИЕ LIGHTGBM НА 5 ФОЛДАХ
# ==========================================

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1
}

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))

print("\nНачало обучения моделей на 5 фолдах...")

# Теперь split сработает идеально, так как y_train — это массив из чистых 0 и 1
for fold, (train_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]

    train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_features)
    val_data = lgb.Dataset(X_va, label=y_va, reference=train_data, categorical_feature=categorical_features)

    model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=3000,
        valid_sets=[train_data, val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / folds.n_splits

    fold_auc = roc_auc_score(y_va, oof_preds[val_idx])
    print(f"-> Фолд {fold + 1} завершен. AUC: {fold_auc:.5f}")

cv_auc = roc_auc_score(y_train, oof_preds)
print(f"\nИтоговый средний Скор Кросс-Валидации (OOF AUC): {cv_auc:.5f}")

# Сборка финального файла
submission = pd.DataFrame({
    'id': test['id'].astype(int),
    'satisfaction': test_preds
})
submission.to_csv('submission.csv', index=False)
print("\nФайл 'submission.csv' готов!")


--- ПРОВЕРКА КОРРЕКТНОСТИ ТАРГЕТА ---
Оригинальные классы, которые нашел энкодер: ['False' 'True']
Распределение закодированных классов в y_train: [389296 310339]
-------------------------------------


Начало обучения моделей на 5 фолдах...
[200]	training's auc: 0.964884	valid_1's auc: 0.959194
[400]	training's auc: 0.967299	valid_1's auc: 0.959433
[600]	training's auc: 0.968755	valid_1's auc: 0.959497
-> Фолд 1 завершен. AUC: 0.95952
[200]	training's auc: 0.965089	valid_1's auc: 0.958134
[400]	training's auc: 0.967512	valid_1's auc: 0.958397
-> Фолд 2 завершен. AUC: 0.95845
[200]	training's auc: 0.96472	valid_1's auc: 0.95979
[400]	training's auc: 0.967	valid_1's auc: 0.960029
[600]	training's auc: 0.968453	valid_1's auc: 0.960112
-> Фолд 3 завершен. AUC: 0.96017
[200]	training's auc: 0.96485	valid_1's auc: 0.959059
-> Фолд 4 завершен. AUC: 0.95929
[200]	training's auc: 0.964871	valid_1's auc: 0.959198
[400]	training's auc: 0.967266	valid_1's auc: 0.959415
[600]	training's auc: 0.968

In [ ]:
!pip install catboost -q  # Установка, если еще не установлен

from catboost import CatBoostClassifier, Pool

# Параметры CatBoost для максимизации AUC
cat_params = {
    'iterations': 3000,
    'learning_rate': 0.05,
    'eval_metric': 'AUC',
    'random_seed': 42,
    'task_type': 'CPU',  # Можно сменить на 'GPU', если в Colab включен T4
    'early_stopping_rounds': 100,
    'verbose': 200
}

oof_preds_cat = np.zeros(len(X_train))
test_preds_cat = np.zeros(len(X_test))
# Автоматически находим ВСЕ категориальные признаки для CatBoost
categorical_features = list(X_train.select_dtypes(include=['category']).columns)
print(f"Зафиксированы категориальные признаки: {categorical_features}")

print("Начало обучения CatBoost на 5 фолдах...")

for fold, (train_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]

    # Для CatBoost категориальные признаки нужно передавать явными индексами или именами
    train_pool = Pool(X_tr, y_tr, cat_features=categorical_features)
    val_pool = Pool(X_va, y_va, cat_features=categorical_features)

    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(train_pool, eval_set=val_pool, use_best_model=True)

    oof_preds_cat[val_idx] = model_cat.predict_proba(X_va)[:, 1]
    test_preds_cat += model_cat.predict_proba(X_test)[:, 1] / folds.n_splits

    fold_auc = roc_auc_score(y_va, oof_preds_cat[val_idx])
    print(f"-> CatBoost Фолд {fold + 1} завершен. AUC: {fold_auc:.5f}")

cv_auc_cat = roc_auc_score(y_train, oof_preds_cat)
print(f"\nИтоговый OOF AUC для CatBoost: {cv_auc_cat:.5f}")


Зафиксированы категориальные признаки: ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'route_id']
Начало обучения CatBoost на 5 фолдах...
0:	test: 0.9346741	best: 0.9346741 (0)	total: 850ms	remaining: 42m 28s
200:	test: 0.9577591	best: 0.9577591 (200)	total: 2m 5s	remaining: 29m 11s
400:	test: 0.9590169	best: 0.9590169 (400)	total: 4m 11s	remaining: 27m 12s
600:	test: 0.9594868	best: 0.9594868 (600)	total: 6m 16s	remaining: 25m 3s
800:	test: 0.9597372	best: 0.9597372 (800)	total: 8m 26s	remaining: 23m 9s
1000:	test: 0.9599126	best: 0.9599126 (1000)	total: 10m 32s	remaining: 21m 3s
1200:	test: 0.9600251	best: 0.9600256 (1199)	total: 12m 40s	remaining: 18m 59s
1400:	test: 0.9601158	best: 0.9601183 (1397)	total: 14m 49s	remaining: 16m 55s
1600:	test: 0.9601891	best: 0.9601898 (1599)	total: 16m 56s	remaining: 14m 48s
1800:	test: 0.9602538	best: 0.9602571 (1785)	total: 19m 4s	remaining: 12m 41s
2000:	test: 0.9603006	best: 0.9603009 (1998)	total: 21m 11s	remaining: 10m 34s
2200:	test

In [ ]:
# ==========================================================
# СОХРАНЕНИЕ ОДНОЧНОГО ЛИДЕРА (ЧИСТЫЙ CATBOOST)
# ==========================================================
print("[INFO] Формирование файла предсказаний для одиночной модели CatBoost...")

# Создаем датафрейм для чистого CatBoost
submission_cat_solo = pd.DataFrame({
    'id': test[id_col].astype(int),
    'satisfaction': test_preds_cat  # берем накопленные предсказания десерта
})

# Сохраняем с понятным и строгим именем
cat_filename = 'submission_catboost_solo_leader.csv'
submission_cat_solo.to_csv(cat_filename, index=False)
print(f"[SUCCESS] Файл одиночного лидера '{cat_filename}' успешно сохранен!")

# Запускаем автоматическое скачивание на ваш компьютер
from google.colab import files
files.download(cat_filename)
print("[INFO] Скачивание файла одиночного лидера на ваш ПК запущено.")


[INFO] Формирование файла предсказаний для одиночной модели CatBoost...
[SUCCESS] Файл одиночного лидера 'submission_catboost_solo_leader.csv' успешно сохранен!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[INFO] Скачивание файла одиночного лидера на ваш ПК запущено.


In [ ]:
# Усредняем предсказания (веса подбираются по качеству моделей, например LightGBM лучший)
final_test_preds = (0.4 * test_preds) + (0.3 * test_preds_cat) + (0.3 * test_preds_xgb)

# Сохраняем финальный победный сабмит
submission_ensemble = pd.DataFrame({
    'id': test['id'].astype(int),
    'satisfaction': final_test_preds
})
submission_ensemble.to_csv('submission_triple_ensemble.csv', index=False)
print("Финальный ансамблевый сабмит 'submission_triple_ensemble.csv' готов к отправке!")
files.download('submission_triple_ensemble.csv')


Финальный ансамблевый сабмит 'submission_triple_ensemble.csv' готов к отправке!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

# ==========================================================
# MASTER BLENDING ENGINE (МЕТОД МАСАИ КАВАМАТЫ)
# ==========================================================

# 1. Собираем все три OOF-предсказания в единую матрицу
# Убедитесь, что все три массива (oof_preds, oof_preds_xgb, oof_preds_cat) посчитаны!
oof_matrix = np.column_stack([oof_preds, oof_preds_xgb, oof_preds_cat])

# Запускаем оптимизатор Hill Climb (шаг 0.005 для максимальной точности)
def optimize_weights_hill_climb(oof_preds_matrix, y_true, iterations=1000, step=0.005):
    n_models = oof_preds_matrix.shape[1]
    best_weights = np.ones(n_models) / n_models

    initial_blend = np.dot(oof_preds_matrix, best_weights)
    best_score = roc_auc_score(y_true, initial_blend)

    print(f"[СТАРТ] AUC при равных долях (по 33%): {best_score:.6f}\n")

    for i in range(iterations):
        improved = False
        for model_idx in range(n_models):
            for change in [-step, step]:
                test_weights = best_weights.copy()
                test_weights[model_idx] += change

                if test_weights[model_idx] < 0:
                    continue

                test_weights /= np.sum(test_weights)

                current_blend = np.dot(oof_preds_matrix, test_weights)
                current_score = roc_auc_score(y_true, current_blend)

                if current_score > best_score:
                    best_score = current_score
                    best_weights = test_weights
                    improved = True
        if not improved:
            break

    print(f"[УСПЕХ] Оптимизация завершена на итерации {i}!")
    print(f"[МАКСИМАЛЬНЫЙ AUC] На кросс-валидации: {best_score:.6f}")
    return best_weights

# Находим идеальные пропорции
best_blend_weights = optimize_weights_hill_climb(oof_matrix, y_train)

# Выводим веса
print("\n--- ИДЕАЛЬНЫЙ РЕЦЕПТ АНСАМБЛЯ ---")
print(f"Вес LightGBM: {best_blend_weights[0]:.4f}")
print(f"Вес XGBoost:  {best_blend_weights[1]:.4f}")
print(f"Вес CatBoost: {best_blend_weights[2]:.4f}")
print("---------------------------------")

# 2. Применяем веса на тестовой выборке
final_ensemble_test_preds = (
    best_blend_weights[0] * test_preds +
    best_blend_weights[1] * test_preds_xgb +
    best_blend_weights[2] * test_preds_cat
)

# 3. Сохраняем и скачиваем честный сабмит
submission_master = pd.DataFrame({
    'id': test['id'].astype(int),
    'satisfaction': final_ensemble_test_preds
})

submission_master.to_csv('submission_shannon_entropy_non_linear_scaled_triple_stack.csv', index=False)
print("\nПобедный файл 'submission_shannon_entropy_non_linear_scaled_triple_stack.csv' сгенерирован и готов к отправке на Kaggle!")

from google.colab import files
files.download('submission_masaya_style.csv')


[СТАРТ] AUC при равных долях (по 33%): 0.960261

[УСПЕХ] Оптимизация завершена на итерации 63!
[МАКСИМАЛЬНЫЙ AUC] На кросс-валидации: 0.960501

--- ИДЕАЛЬНЫЙ РЕЦЕПТ АНСАМБЛЯ ---
Вес LightGBM: 0.2119
Вес XGBoost:  0.0721
Вес CatBoost: 0.7159
---------------------------------

Победный файл 'submission_masaya_style.csv' сгенерирован и готов к отправке на Kaggle!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>